# T01 — Environmental field loader + Stage III interface (manual check)

Scope of this notebook (per `tasks.md` T01 and the 2026-09-22 task instructions):

- run the T01 loader (`iste.environment.load_field_dataset`) against the real
  `data/ocean.nc` and `data/wind.nc`, and print the required diagnostics: dimensions,
  timestamp range, coordinate directions, and missing counts;
- show that the loader is layout-agnostic (same function, different dimension names
  and latitude order, no hard-coded NetCDF layout);
- show that `EnvironmentalFieldProvider` (the Stage III `U/K/grad_K/div_U/validity`
  interface from `README.md`) is defined but **not implemented** — it cannot be
  instantiated, and no particle propagation, interpolation, KDE, likelihood, forcing
  ensemble, or physics has been added. That is intentionally deferred to T02+.

This notebook does not compute or claim any scientific result — it is a manual
inspection aid only, per `CLAUDE.md` (`.ipynb` = experiments/plots/debugging/manual
validation; `.py` = reusable logic).

In [ ]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np

from iste.environment import EnvironmentalFieldProvider, load_field_dataset

OCEAN_PATH = os.path.join(REPO_ROOT, "data", "ocean.nc")
WIND_PATH = os.path.join(REPO_ROOT, "data", "wind.nc")
print("repo root:", REPO_ROOT)

## Ocean dataset (`data/ocean.nc`) — dims, timestamp range, coordinate direction, missing counts

In [ ]:
ocean = load_field_dataset(
    OCEAN_PATH, ["uo", "vo"], time_dim="time", lat_dim="latitude", lon_dim="longitude"
)
ocean_meta = ocean.describe()
print(ocean_meta.as_table())

**Expected (from T00 inspection, `progress.md`):** `time` n=336,
`2019-10-01T00:00:00 -> 2019-10-14T23:00:00`, `latitude` n=149 ascending, `longitude`
n=216, `uo`/`vo` shape `(336, 149, 216)`, missing count `2974608` for each (land/no-data
cells masked from the raw `1e20` fill sentinel).

## Wind dataset (`data/wind.nc`) — different dimension names, opposite latitude order

In [ ]:
wind = load_field_dataset(
    WIND_PATH, ["u10", "v10"], time_dim="valid_time", lat_dim="latitude", lon_dim="longitude"
)
wind_meta = wind.describe()
print(wind_meta.as_table())

**Expected:** `valid_time` n=336, same range as ocean, `latitude` n=25
**descending** (opposite of the ocean file), `longitude` n=36, `u10`/`v10` shape
`(336, 25, 36)`, missing count `0` (no fill values found in this file).

In [ ]:
# Cross-check: timestamps must align exactly between the two files.
print("timestamps equal:", bool(np.array_equal(ocean.time, wind.time)))
print("ocean lat ascending:", ocean.lat_ascending, " | wind lat ascending:", wind.lat_ascending)

## Loader is layout-agnostic (no hard-coded NetCDF layout)

The same `load_field_dataset` function is called above with two different sets of
dimension names (`time`/`latitude`/`longitude` vs `valid_time`/`latitude`/`longitude`)
and two different latitude orderings, with nothing project-specific hard-coded inside
the function. `tests/test_environment.py::TestLoaderIsLayoutAgnostic` additionally proves
this against a synthetic file using entirely different dimension names (`T`/`Y`/`X`).

## Stage III interface exists but is intentionally not implemented

`EnvironmentalFieldProvider` declares `U`, `K`, `grad_K`, `div_U`, `validity` as required
by `README.md`, but every method is abstract. This cell shows the interface cannot be
instantiated (no accidental/partial physics), and that a full stub subclass compiles but
still raises `NotImplementedError` when called — confirming nothing beyond the T01
contract has been built.

In [ ]:
try:
    EnvironmentalFieldProvider()
except TypeError as exc:
    print("Cannot instantiate the bare interface, as expected:")
    print(" ", exc)


class StubProvider(EnvironmentalFieldProvider):
    def U(self, x, t):
        raise NotImplementedError

    def K(self, x, t):
        raise NotImplementedError

    def grad_K(self, x, t):
        raise NotImplementedError

    def div_U(self, x, t):
        raise NotImplementedError

    def validity(self, x, t):
        raise NotImplementedError


provider = StubProvider()
try:
    provider.U(np.zeros(2), ocean.time[0])
except NotImplementedError:
    print("StubProvider.U(...) correctly raises NotImplementedError (T02/T03 not built yet).")

## Summary / known limitations

- Loader + diagnostics above match the T00 inspection findings; no discrepancy found.
- `EnvironmentalFieldProvider` fixes the Stage III interface contract only. It has **no**
  working implementation of `U`, `K`, `grad_K`, `div_U`, or `validity` yet — those need,
  respectively: spatial/temporal interpolation (T02), finite-difference operators (T03),
  an explicitly supplied `K` definition (undecided), and an explicit windage-combination
  decision for `U` (undecided). None of these have been invented here.
- No particle propagation, KDE, likelihood, forcing-ensemble perturbation, SMC, MCMC, or
  AIS code exists in this repository.